In [ ]:
# ==========================================================
# SLiMS ANTI-FRAGILE PDF SCRAPER v2 (ENHANCED)
# ----------------------------------------------------------
# Features:
# - Attachment-section first parsing (MOST IMPORTANT FIX)
# - Multi fallback extraction strategy
# - Safe validation for fstream links (fid + bid)
# - Resume support (skip existing files)
# - tqdm progress
# - Random delay (polite crawling)
# - Failure logging with reason
# ==========================================================

import pandas as pd
import requests
from bs4 import BeautifulSoup
from pathlib import Path
from tqdm.auto import tqdm
import time
import random
import re

# =========================
# CONFIG
# =========================

CSV_FILE = "docid.csv"
OUTPUT_DIR = "pdf"

BASE_URL = "https://repositori.untidar.ac.id"

MIN_DELAY = 2
MAX_DELAY = 6

Path(OUTPUT_DIR).mkdir(exist_ok=True)

# =========================
# SESSION (faster + stable)
# =========================

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (compatible; ResearchBot/1.0)"
})

# =========================
# LOAD DOCID
# =========================

df = pd.read_csv(CSV_FILE)
docids = df["docid"].astype(str).tolist()

print("Total docid:", len(docids))

# =========================
# VALIDATION FUNCTION
# =========================

def is_valid_fstream(href: str) -> bool:
    """Validasi link PDF SLiMS (harus fstream + fid + bid)"""
    return bool(href and "p=fstream" in href and "fid=" in href and "bid=" in href)

# =========================
# NORMALIZE URL
# =========================

def normalize_url(href: str) -> str:
    """Handle ./index.php relative link"""
    if href.startswith("./"):
        href = href[2:]
    return BASE_URL + "/" + href

# =========================
# CORE EXTRACTION ENGINE
# =========================

def extract_pdf_link(soup):
    """
    STRATEGY ORDER (VERY IMPORTANT):
    1. Attachment section (most reliable)
    2. openPopUp class scan
    3. global fstream scan
    4. regex fallback
    """

    # --------------------------------------------------
    # STRATEGY 1: ATTACHMENT SECTION (BEST SOURCE)
    # --------------------------------------------------
    attachment = soup.find(id="attachment")

    if attachment:
        for a in attachment.find_all("a", href=True):
            href = a["href"]
            if is_valid_fstream(href):
                return href

    # --------------------------------------------------
    # STRATEGY 2: CLASS-BASED (openPopUp)
    # --------------------------------------------------
    for a in soup.select("a.openPopUp"):
        href = a.get("href", "")
        if is_valid_fstream(href):
            return href

    # --------------------------------------------------
    # STRATEGY 3: GLOBAL SCAN (robust fallback)
    # --------------------------------------------------
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if is_valid_fstream(href):
            return href

    # --------------------------------------------------
    # STRATEGY 4: REGEX (LAST RESORT)
    # --------------------------------------------------
    html = str(soup)

    match = re.search(
        r"index\.php\?p=fstream[^\"']+fid=\d+[^\"']+bid=\d+",
        html
    )

    if match:
        return match.group(0)

    return None

# =========================
# DOWNLOAD ENGINE
# =========================

success = 0
failed = 0
skipped = 0

failed_log = []

for docid in tqdm(docids, desc="Downloading SLiMS PDFs"):

    output_file = Path(OUTPUT_DIR) / f"{docid}.pdf"

    # -------------------------
    # RESUME SUPPORT
    # -------------------------
    if output_file.exists():
        skipped += 1
        continue

    try:
        # -------------------------
        # STEP 1: GET DETAIL PAGE
        # -------------------------
        url = f"{BASE_URL}/index.php?p=show_detail&id={docid}"
        r = session.get(url, timeout=30)
        r.raise_for_status()

        soup = BeautifulSoup(r.text, "html.parser")

        # -------------------------
        # DEBUG CHECK (optional)
        # -------------------------
        # print("DOCID:", docid, "HAS FULLTEXT:", "fstream" in r.text)

        # -------------------------
        # STEP 2: EXTRACT PDF LINK
        # -------------------------
        href = extract_pdf_link(soup)

        if not href:
            failed += 1
            failed_log.append((docid, "NO_FSTREAM_FOUND"))
            continue

        pdf_url = normalize_url(href)

        # -------------------------
        # STEP 3: DOWNLOAD PDF
        # -------------------------
        pdf = session.get(pdf_url, timeout=120)
        pdf.raise_for_status()

        # sanity check PDF header
        if not pdf.content.startswith(b"%PDF"):
            failed += 1
            failed_log.append((docid, "NOT_PDF_RESPONSE"))
            continue

        # save file
        with open(output_file, "wb") as f:
            f.write(pdf.content)

        success += 1

        # polite delay
        time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))

    except Exception as e:
        failed += 1
        failed_log.append((docid, f"ERROR:{str(e)[:50]}"))

        time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))

# =========================
# SAVE FAILURE LOG
# =========================

pd.DataFrame(failed_log, columns=["docid", "reason"]).to_csv(
    "failed_log.csv",
    index=False
)

# =========================
# SUMMARY
# =========================

print("\n====================")
print("DONE")
print("====================")
print("SUCCESS :", success)
print("FAILED  :", failed)
print("SKIPPED :", skipped)
print("LOG FILE -> failed_log.csv")

Total docid: 1


Downloading:   0%|          | 0/1 [00:00<?, ?it/s]


DONE
SUCCESS: 0
FAILED : 1
SKIPPED: 0
FAILED LOG -> failed_docids.csv
